In [1]:
import pandas as pd
import json
from sqlalchemy import create_engine

In [3]:
# 1. Create the Engine
# This creates a local file named 'yelp.db' in your current folder
print("Creating database engine...")
engine = create_engine('sqlite:///yelp.db')

# 2. Process the smaller files in one go
small_files = ['business', 'checkin', 'tip']

for file in small_files:
    print(f"Loading {file}.json...")
    df = pd.read_json(f'yelp_academic_dataset_{file}.json', lines=True, encoding='utf-8')
    
    # Drop nested JSON columns from the business table so SQL can read it
    if file == 'business':
        df = df.drop(columns=['attributes', 'hours'], errors='ignore')
        
    # Send to the SQL engine
    df.to_sql(file, engine, if_exists='replace', index=False)
    print(f"--> Saved {file} to yelp.db")

Creating database engine...
Loading business.json...
--> Saved business to yelp.db
Loading checkin.json...
--> Saved checkin to yelp.db
Loading tip.json...
--> Saved tip to yelp.db


In [5]:
# We reuse the same 'engine' variable from Cell 1

# 3. Process the massive review file via Chunking
print("Starting streaming process for review.json...")

reader = pd.read_json('yelp_academic_dataset_review.json', lines=True, chunksize=100000, encoding='utf-8')

for i, chunk in enumerate(reader):
    # Drop the heavy text column to save gigabytes of space
    chunk = chunk.drop(columns=['text'], errors='ignore')
    
    # First chunk replaces the table, the rest append to it
    mode = 'replace' if i == 0 else 'append'
    chunk.to_sql('review', engine, if_exists=mode, index=False)
    
    print(f"--> Processed review chunk {i+1} ({(i+1)*100000} rows)")

print("Data engineering complete! Your yelp.db is fully loaded.")

Starting streaming process for review.json...
--> Processed review chunk 1 (100000 rows)
--> Processed review chunk 2 (200000 rows)
--> Processed review chunk 3 (300000 rows)
--> Processed review chunk 4 (400000 rows)
--> Processed review chunk 5 (500000 rows)
--> Processed review chunk 6 (600000 rows)
--> Processed review chunk 7 (700000 rows)
--> Processed review chunk 8 (800000 rows)
--> Processed review chunk 9 (900000 rows)
--> Processed review chunk 10 (1000000 rows)
--> Processed review chunk 11 (1100000 rows)
--> Processed review chunk 12 (1200000 rows)
--> Processed review chunk 13 (1300000 rows)
--> Processed review chunk 14 (1400000 rows)
--> Processed review chunk 15 (1500000 rows)
--> Processed review chunk 16 (1600000 rows)
--> Processed review chunk 17 (1700000 rows)
--> Processed review chunk 18 (1800000 rows)
--> Processed review chunk 19 (1900000 rows)
--> Processed review chunk 20 (2000000 rows)
--> Processed review chunk 21 (2100000 rows)
--> Processed review chunk 

In [8]:
# We reuse the same 'engine' variable from the previous cells

# 4. Process the massive user file via Chunking
print("Starting streaming process for user.json...")

# Chunk size of 100,000 is still perfect for your 16GB of RAM
user_reader = pd.read_json('yelp_academic_dataset_user.json', lines=True, chunksize=100000, encoding='utf-8')

for i, chunk in enumerate(user_reader):
    # Drop the massive 'friends' column to save database space
    chunk = chunk.drop(columns=['friends'], errors='ignore')
    
    # First chunk replaces the table, the rest append to it
    mode = 'replace' if i == 0 else 'append'
    chunk.to_sql('user', engine, if_exists=mode, index=False)
    
    print(f"--> Processed user chunk {i+1} ({(i+1)*100000} rows)")

print("User data engineering complete! All 5 tables are now in yelp.db.")

Starting streaming process for user.json...
--> Processed user chunk 1 (100000 rows)
--> Processed user chunk 2 (200000 rows)
--> Processed user chunk 3 (300000 rows)
--> Processed user chunk 4 (400000 rows)
--> Processed user chunk 5 (500000 rows)
--> Processed user chunk 6 (600000 rows)
--> Processed user chunk 7 (700000 rows)
--> Processed user chunk 8 (800000 rows)
--> Processed user chunk 9 (900000 rows)
--> Processed user chunk 10 (1000000 rows)
--> Processed user chunk 11 (1100000 rows)
--> Processed user chunk 12 (1200000 rows)
--> Processed user chunk 13 (1300000 rows)
--> Processed user chunk 14 (1400000 rows)
--> Processed user chunk 15 (1500000 rows)
--> Processed user chunk 16 (1600000 rows)
--> Processed user chunk 17 (1700000 rows)
--> Processed user chunk 18 (1800000 rows)
--> Processed user chunk 19 (1900000 rows)
--> Processed user chunk 20 (2000000 rows)
User data engineering complete! All 5 tables are now in yelp.db.
